In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "ev_battery_health.csv"
file_dictionary = "data_dictionary .csv"
# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "vinay24baghira/ev-battery-health-and-degradation-dataset",
  file_path
)

ddic = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "vinay24baghira/ev-battery-health-and-degradation-dataset",
  file_dictionary
)




In [ ]:
print(df.dtypes)
df.describe(include='all')

In [ ]:
# 1 # Check for columns with missing data
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0]

if not missing_data.empty:
    print("Columns with missing data and their counts:")
    print(missing_data)
else:
    print("No missing data found in any column.")



In [ ]:
# 2 ##We can not have these rows in our data because there is not enough time/count data in the reading if all three are missing. 5 rows removed as we can't use those readings.  Forward thinking is time will be important and we can still substitute battery_age_days and cycle count to estimate when a reading happened
columns_to_check = ['observation_date', 'battery_age_days', 'cycle_count']

# Record starting shape
initial_shape = df.shape

# Drop rows where all three columns are null
df = df.dropna(subset=columns_to_check, how='all')

# Print changes
print(f"Original shape: {initial_shape}")
print(f"New shape: {df.shape}")
print(f"Removed rows: {initial_shape[0] - df.shape[0]}")

In [ ]:
#3 
def get_non_null_value(battery_id, col_name):
    """
    Looks up a non-null value for a given battery_id and column name from df.
    """
    values = df[df['battery_id'] == battery_id][col_name].dropna()
    if not values.empty:
        return values.iloc[0]
    return np.nan

# Group the DataFrame by 'battery_id' to build an efficient lookup dictionary for each specified column
print("Populating gaps in the dataset for specified columns...")

# Specific columns to fill, these rows related to the battery pack and not the reading. So just for completelness.
columns_to_fill = ["chemistry", "manufacturer", "vehicle_model", "pack_capacity_kwh", "manufacture_date", "climate_zone"]

# Iterate and apply imputation using other rows of the same battery_id
for col in columns_to_fill:
    if col in df.columns:
        # Create a lookup mapping battery_id to its first non-null value for this column
        lookup_map = df.groupby('battery_id')[col].first()

        # Get indices where the column is null
        null_mask = df[col].isnull()

        # Map the battery_id of null rows to the lookup map
        imputed_values = df.loc[null_mask, 'battery_id'].map(lookup_map)

        # Fill the missing values
        df.loc[null_mask, col] = df.loc[null_mask, col].fillna(imputed_values)

# Verify new missing value counts for the selected columns
remaining_missing = df[columns_to_fill].isnull().sum()
print("\nRemaining missing data counts for targeted columns:")
print(remaining_missing)

In [ ]:
# 4 # Define columns for each table, there are two tables if we think of the flat table as containing two objects, the battery and its readings.
battery_cols = ["battery_id", "chemistry", "manufacturer", "vehicle_model", "pack_capacity_kwh", "manufacture_date", "climate_zone"]
reading_cols = ["observation_id", "battery_id", "observation_date", "battery_age_days", "cycle_count", "avg_daily_distance_km", "avg_charging_power_kw", "fast_charge_ratio", "avg_depth_of_discharge_pct", "avg_state_of_charge_pct", "charge_cycles_per_month", "total_energy_throughput_kwh", "avg_ambient_temp_c", "humidity_pct", "altitude_m", "internal_resistance_mohm", "voltage_avg_v", "current_avg_a", "charge_time_avg_min", "peak_temp_during_fast_charge_c", "capacity_fade_pct", "service_count", "last_service_date", "bms_firmware_version", "recall_flag", "state_of_health_pct", "remaining_useful_life_cycles", "failure_risk_label", "degradation_category"]

# Create BatteryTable with unique battery_id entries
BatteryTable = df[battery_cols].drop_duplicates(subset=["battery_id"]).reset_index(drop=True)

# Create ReadingTable with unique observation_id entries
# Note: battery_id is included as a foreign key to maintain the relationship
ReadingTable = df[reading_cols].drop_duplicates(subset=["observation_id"]).reset_index(drop=True)

# Display table summaries and previews
print(f"BatteryTable shape: {BatteryTable.shape}")
print(f"ReadingTable shape: {ReadingTable.shape}")

print("\nBatteryTable preview:")
display(BatteryTable.head())

print("\nReadingTable preview:")
display(ReadingTable.head())

In [ ]:
# 5. Ensure observation_date is in datetime64 format ready for sort
ReadingTable['observation_date'] = pd.to_datetime(ReadingTable['observation_date'])

# Print the count of missing cycle_count values BEFORE the operation
initial_missing = ReadingTable['cycle_count'].isnull().sum()
print(f"Missing 'cycle_count' values BEFORE interpolation: {initial_missing}")

# 2. Sort the table chronologically by battery_id and observation_date
ReadingTable = ReadingTable.sort_values(by=['battery_id', 'observation_date']).reset_index(drop=True)

# 3 & 4. Group by battery_id, perform linear interpolation, and apply ffill/bfill for boundary cases
def interpolate_battery_cycles(group):
    # Interpolate over the sorted sequence within each battery group
    group['cycle_count'] = group['cycle_count'].interpolate(method='linear').ffill().bfill()
    return group

# Apply the group-by operation using include_groups=False to avoid DeprecationWarnings
#ReadingTable = ReadingTable.groupby('battery_id', group_keys=False).apply(interpolate_battery_cycles, include_groups=False)

# 5. Print the count of missing cycle_count values AFTER the operation
final_missing = ReadingTable['cycle_count'].isnull().sum()
print(f"Missing 'cycle_count' values AFTER interpolation: {final_missing}")

In [ ]:
display(ReadingTable.head())
display(BatteryTable.head())

In [ ]:
duplicate_count = ReadingTable.duplicated().sum()# No duplicate to sort out
duplicate_count

In [ ]:
# 6 # Calculate the percentage of missing values for each column in ReadingTable
missing_pct = (ReadingTable.isnull().mean() * 100).sort_values(ascending=False)

# Plotting the missing percentage as a horizontal bar graph
plt.figure(figsize=(10, 8))
missing_pct.plot(kind='barh')
plt.xlabel('% missing')
plt.title('Missingness by column in ReadingTable')
plt.tight_layout()

# Save the figure to a PNG file
plt.savefig('missingness_by_column.png', dpi=300)
plt.show()

In [ ]:
ddic


In [ ]:
#7 
def impute_battery_readings(df):
    """
    Fills missing values in ReadingTable using a moving average of past observations within each battery_id group.
    """
    # Create a working copy of the DataFrame
    imputed_df = df.copy()

    # Target features specified by the user
    target_features = [
        'capacity_fade_pct',
        'internal_resistance_mohm',
        'total_energy_throughput_kwh',
        'fast_charge_ratio'
    ]

    # Step 1: Ensure date column is datetime format and sort chronologically within each battery_id
    imputed_df['observation_date'] = pd.to_datetime(imputed_df['observation_date'])
    imputed_df = imputed_df.sort_values(by=['battery_id', 'observation_date']).reset_index(drop=True)

    # Step 2: Define a function to perform rolling imputation on each battery group
    def process_group(group):
        for feature in target_features:
            if feature in group.columns:
                # Calculate moving average of past observations (closed on left to use only historical records)
                # We use a window size of 3 as a baseline (can adjust if required), min_periods=1
                rolling_avg = group[feature].shift(1).rolling(window=3, min_periods=1).mean()

                # Impute missing values using the moving average
                group[feature] = group[feature].fillna(rolling_avg)

                # Fallback ffill and bfill for initial boundary conditions or persistent NaNs
                group[feature] = group[feature].ffill().bfill()
        return group

    # Step 3: Apply the grouped operation strictly by battery_id while preserving battery_id column in the output DataFrame
    # We set group_keys=True so that the grouped keys are maintained, or we can restore the grouped index column.
    # Using group_keys=False and not using include_groups=False (keeping default or manually handling) ensures battery_id is retained.
    imputed_df = imputed_df.groupby('battery_id', group_keys=False).apply(process_group)

    return imputed_df

# Run the imputation pipeline on ReadingTable
print("Missing counts before imputation:")
print(ReadingTable[['capacity_fade_pct', 'internal_resistance_mohm', 'total_energy_throughput_kwh', 'fast_charge_ratio']].isnull().sum())

ReadingTable = impute_battery_readings(ReadingTable)

print("\nMissing counts after imputation:")
print(ReadingTable[['capacity_fade_pct', 'internal_resistance_mohm', 'total_energy_throughput_kwh', 'fast_charge_ratio']].isnull().sum())

In [ ]:
#8 # Export the cleaned ReadingTable DataFrame to a CSV file check it in excel.
ReadingTable.to_csv('ReadingTable_cleaned.csv', index=False)
print("ReadingTable successfully saved to 'ReadingTable_cleaned.csv'")
print(f"ReadingTable shape: {ReadingTable.shape}")

# Export the cleaned ReadingTable DataFrame to a CSV file
BatteryTable.to_csv('BatteryTable_cleaned.csv', index=False)
print("BatteryTable successfully saved to 'BatteryTable_cleaned.csv'")
print(f"BatteryTable shape: {BatteryTable.shape}")

In [ ]:
# 9 #create heat maps for initial corralation detection to the target State of Health (SoH)
## Given in the dataset that this is the target.    But there maybe other targets linked to it.
def plot_chemistry_heatmaps(df, target_col='state_of_health_pct'):
    chemistries = ['LMO', 'LFP', 'NMC', 'NCA']
    features = [
        'capacity_fade_pct',
        'internal_resistance_mohm',
        'total_energy_throughput_kwh',
        'fast_charge_ratio',
        target_col
    ]

    # Create a grid of 1 row x 4 columns for comparison
    fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=True)

    for i, chem in enumerate(chemistries):
        # Filter dataset by chemistry type
        chem_df = df[df['chemistry'] == chem][features]
        corr_matrix = chem_df.corr()

        # Plot individual heatmap
        sns.heatmap(
            corr_matrix,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            ax=axes[i],
            cbar=(i == 3), # Show colorbar only on the last plot
            vmin=-1,
            vmax=1
        )
        axes[i].set_title(f'Chemistry: {chem}')

    plt.tight_layout()
    plt.show()

In [ ]:
# 10. Merge ReadingTable with BatteryTable using battery_id
merged_df = pd.merge(
    ReadingTable,
    BatteryTable[['battery_id', 'chemistry', 'manufacturer']],  # Pull only the chemistry and manufacturer columns needed
    on='battery_id',
    how='inner',
)

In [ ]:
numeric_cols = merged_df.select_dtypes(include=[np.number]).columns.tolist()
plt.figure(figsize=(14, 11))
sns.heatmap(merged_df[numeric_cols].corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Correlation matrix — numeric features')
plt.tight_layout()
plt.show()

In [ ]:
##11 added statistic on the boundary of other SoH levels.
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Configure visual settings
plt.figure(figsize=(18, 5))

# 1. State of Health vs Average Remaining Useful Life Cycles
# Since SoH is continuous, we group by integer SoH levels to find the average RUL
merged_df['soh_int'] = merged_df['state_of_health_pct'].round().astype(int)
avg_rul_by_soh = merged_df.groupby('soh_int')['remaining_useful_life_cycles'].mean().reset_index()

plt.subplot(1, 3, 1)
sns.lineplot(data=avg_rul_by_soh, x='soh_int', y='remaining_useful_life_cycles', marker='o', color='blue', linewidth=2.5)
plt.title('Average Remaining RUL Cycles by State of Health', fontsize=12)
plt.xlabel('State of Health (%)')
plt.ylabel('Average Remaining Useful Life (Cycles)')
plt.grid(True, linestyle='--', alpha=0.6)

# 2. State of Health vs Failure Risk Label
# Boxplot is highly suitable for continuous vs categorical to show distributions clearly
plt.subplot(1, 3, 2)
risk_order = ['Low', 'Medium', 'High']
sns.boxplot(data=merged_df, x='failure_risk_label', y='state_of_health_pct', order=risk_order, palette='Oranges', hue='failure_risk_label', legend=False)
plt.title('State of Health by Failure Risk Label', fontsize=12)
plt.xlabel('Failure Risk Label')
plt.ylabel('State of Health (%)')
plt.grid(True, linestyle='--', alpha=0.6)

# 3. State of Health vs Degradation Category
plt.subplot(1, 3, 3)
degradation_order = ['Normal', 'Accelerated', 'Critical']
sns.boxplot(data=merged_df, x='degradation_category', y='state_of_health_pct', order=degradation_order, palette='Reds', hue='degradation_category', legend=False)
plt.title('State of Health by Degradation Category', fontsize=12)
plt.xlabel('Degradation Category')
plt.ylabel('State of Health (%)')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
##12
import pandas as pd

# Group by failure_risk_label and calculate quartiles for state_of_health_pct
risk_quartiles = merged_df.groupby('failure_risk_label')['state_of_health_pct'].quantile([0.25, 0.50, 0.75]).unstack()
risk_quartiles.columns = ['Lower Quartile (25%)', 'Median (50%)', 'Upper Quartile (75%)']
risk_quartiles = risk_quartiles.reindex(['Low', 'Medium', 'High'])

print("--- State of Health Boundaries by Failure Risk Label ---")
display(risk_quartiles)

# Group by degradation_category and calculate quartiles for state_of_health_pct
degradation_quartiles = merged_df.groupby('degradation_category')['state_of_health_pct'].quantile([0.25, 0.50, 0.75]).unstack()
degradation_quartiles.columns = ['Lower Quartile (25%)', 'Median (50%)', 'Upper Quartile (75%)']
degradation_quartiles = degradation_quartiles.reindex(['Normal', 'Accelerated', 'Critical'])

print("\n--- State of Health Boundaries by Degradation Category ---")
display(degradation_quartiles)

In [ ]:
##13
def plot_global_heatmap(df, target_col='state_of_health_pct'):
    features = [
        'capacity_fade_pct',
        'internal_resistance_mohm',
        'total_energy_throughput_kwh',
        'fast_charge_ratio',
        'avg_charging_power_kw',
        target_col,
    ]

    plt.figure(figsize=(8, 6))

    # Calculate correlation matrix for numerical features
    corr = df[features].corr()

    # Draw single heatmap
    sns.heatmap(
        corr,
        annot=True,
        fmt='.2f',
        cmap='coolwarm',
        vmin=-1,
        vmax=1,
        linewidths=0.5,
    )

    plt.title('Global Correlation Matrix (All Battery Readings)', fontsize=14)
    plt.tight_layout()
    plt.show()


# Execute global heatmap
plot_global_heatmap(merged_df)

In [ ]:
##14
def plot_chemistry_heatmaps(df, target_col='state_of_health_pct'):
    chemistries = ['LMO', 'LFP', 'NMC', 'NCA']
    features = [
        'capacity_fade_pct',
        'internal_resistance_mohm',
        'total_energy_throughput_kwh',
        'fast_charge_ratio',
        target_col,
    ]

    # Create a 1-row by 4-column figure grid
    fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=True)

    for i, chem in enumerate(chemistries):
        # Filter merged data for the current chemistry
        chem_data = df[df['chemistry'] == chem][features]

        # Handle edge case: Ensure group is not empty before computing correlation
        if len(chem_data) > 0:
            corr = chem_data.corr()

            sns.heatmap(
                corr,
                annot=True,
                fmt='.2f',
                cmap='coolwarm',
                ax=axes[i],
                cbar=(
                    i == 3
                ),  # Display colorbar only on the far right chart for cleanliness
                vmin=-1,
                vmax=1,
            )
            axes[i].set_title(f'Chemistry: {chem}', fontsize=12)
        else:
            axes[i].set_title(f'{chem} (No Data)', fontsize=12)

    plt.suptitle(
        'Feature Correlation by Cell Chemistry', fontsize=16, y=1.05
    )
    plt.tight_layout()
    plt.show()


# Execute chemistry subplots
plot_chemistry_heatmaps(merged_df)